# [0] 라이브러리 설정 및 데이터 로드
- 문헌 수집: [Web of Science](https://www.webofscience.com)
- 검색 query: ("generative AI" OR "large language model" OR "LLM" OR "ChatGPT") AND (finance OR banking OR fintech)

In [2]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# 1. 데이터 로드 (컬럼명 AB, TI, AU 활용) 
raw_df = pd.read_excel('WOS_data.xls')
print(f"총 데이터 수: {len(raw_df)}")
print(f"컬럼 개수: {len(raw_df.columns)}")
print(f"컬럼명: {raw_df.columns.tolist()}")

총 데이터 수: 608
컬럼 개수: 72
컬럼명: ['Publication Type', 'Authors', 'Book Authors', 'Book Editors', 'Book Group Authors', 'Author Full Names', 'Book Author Full Names', 'Group Authors', 'Article Title', 'Source Title', 'Book Series Title', 'Book Series Subtitle', 'Language', 'Document Type', 'Conference Title', 'Conference Date', 'Conference Location', 'Conference Sponsor', 'Conference Host', 'Author Keywords', 'Keywords Plus', 'Abstract', 'Addresses', 'Affiliations', 'Reprint Addresses', 'Email Addresses', 'Researcher Ids', 'ORCIDs', 'Funding Orgs', 'Funding Name Preferred', 'Funding Text', 'Cited References', 'Cited Reference Count', 'Times Cited, WoS Core', 'Times Cited, All Databases', '180 Day Usage Count', 'Since 2013 Usage Count', 'Publisher', 'Publisher City', 'Publisher Address', 'ISSN', 'eISSN', 'ISBN', 'Journal Abbreviation', 'Journal ISO Abbreviation', 'Publication Date', 'Publication Year', 'Volume', 'Issue', 'Part Number', 'Supplement', 'Special Issue', 'Meeting Abstract', 'Start

In [3]:
# 필요한 컬럼만 추출
article_df = raw_df[['Article Title', 'Authors', 'Abstract']].copy()

# 컬럼명 변경
article_df.columns = ['Title', 'Author', 'Abstract']

# 초록(Abstract)이 없는 데이터 제거
article_df = article_df.dropna(subset=['Abstract']).reset_index(drop=True)

# 확인
print(article_df.shape)
article_df.head()

(602, 3)


,Title,Author,Abstract
0,The Odyssey of robots.txt Governance: Measurin...,"Cui, J; Zha, MM; Wang, XF; Liao, XJ",Web content is an essential element for large ...
1,Evaluation of a Large Language Model on the Am...,"Ramgopal, S; Varma, S; Gorski, JK; Kester, KM;...","BackgroundLarge language models (LLMs), includ..."
2,HDLGen-ChatGPT Case Study: RISC-V Processor VH...,"Morgan, F; Byrne, JP; Bupathi, A; George, R; E...",This paper presents the open source HDLGen-Cha...
3,Experience with Large Language Model Applicati...,"Yu, L; Alégroth, E; Chatzipetrou, P; Gorschek, T",Large Language Models (LLMs) offer promising c...
4,Large Language Model Agents for Investment Man...,"Saha, P; Lyu, JR; Saxena, A; Zhao, TJ; Mehta, D",Recent advances in Large Language Models (LLMs...


#

# [1] 전처리

## 1-1. 전처리 라이브러리 및 기본 설정

In [4]:
import re
import nltk
import pandas as pd

from collections import Counter
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# nltk 다운로드
nltk.download('stopwords')
nltk.download('wordnet')

# 표제어 추출기
lemmatizer = WordNetLemmatizer()

# 기본 영어 불용어
base_stopwords = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/gomdori/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /Users/gomdori/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


## 1-2. 1차 전처리
`article_df`
- 결측치 처리, 소문자 변환, 영문 외 문자 제거, 토큰화
- 기본 불용어 제거
- 2자 이하 짧은 단어 제거
- 표제어 추출

In [5]:
def initial_clean_text(text):

    # 결측치 처리
    if pd.isna(text):
        return ""

    # 소문자 변환
    text = text.lower()

    # 영문자 외 제거
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)

    # 공백 정리
    text = re.sub(r'\s+', ' ', text).strip()

    # 토큰화
    words = text.split()

    # 기본 불용어 제거 + 짧은 단어 제거
    words = [
        w for w in words
        if w not in base_stopwords and len(w) > 2
    ]

    # 표제어 추출
    words = [lemmatizer.lemmatize(w) for w in words]

    return words

# 1차 전처리 결과 저장
article_df['Initial_Tokens'] = article_df['Abstract'].apply(initial_clean_text)

print(f"1차 전처리 완료: {len(article_df)}건")
article_df.head()

1차 전처리 완료: 602건


,Title,Author,Abstract,Initial_Tokens
0,The Odyssey of robots.txt Governance: Measurin...,"Cui, J; Zha, MM; Wang, XF; Liao, XJ",Web content is an essential element for large ...,"[web, content, essential, element, large, lang..."
1,Evaluation of a Large Language Model on the Am...,"Ramgopal, S; Varma, S; Gorski, JK; Kester, KM;...","BackgroundLarge language models (LLMs), includ...","[backgroundlarge, language, model, llm, includ..."
2,HDLGen-ChatGPT Case Study: RISC-V Processor VH...,"Morgan, F; Byrne, JP; Bupathi, A; George, R; E...",This paper presents the open source HDLGen-Cha...,"[paper, present, open, source, hdlgen, chatgpt..."
3,Experience with Large Language Model Applicati...,"Yu, L; Alégroth, E; Chatzipetrou, P; Gorschek, T",Large Language Models (LLMs) offer promising c...,"[large, language, model, llm, offer, promising..."
4,Large Language Model Agents for Investment Man...,"Saha, P; Lyu, JR; Saxena, A; Zhao, TJ; Mehta, D",Recent advances in Large Language Models (LLMs...,"[recent, advance, large, language, model, llm,..."


## 1-3. 단어 빈도 및 문헌 등장 비율 확인

In [6]:
# 전체 단어 빈도 계산
all_words = []

for tokens in article_df['Initial_Tokens']:
    all_words.extend(tokens)

word_freq = Counter(all_words)

# 문헌 등장 비율 계산
doc_freq = Counter()

for tokens in article_df['Initial_Tokens']:
    unique_words = set(tokens)
    doc_freq.update(unique_words)

# 데이터프레임 생성
freq_df = pd.DataFrame({
    'word': list(doc_freq.keys()),
    'doc_count': list(doc_freq.values())
})

# 문헌 등장 비율(%)
freq_df['doc_ratio'] = (
    freq_df['doc_count'] / len(article_df)
) * 100

# 전체 빈도 추가
freq_df['total_freq'] = freq_df['word'].map(word_freq)

# 정렬
freq_df = freq_df.sort_values(
    by='doc_ratio',
    ascending=False
)

# 상위 단어 확인
freq_df.head(30)

,word,doc_count,doc_ratio,total_freq
8,model,490,81.395349,1296
7,language,417,69.269103,649
83,large,406,67.441860,515
24,llm,367,60.963455,1312
310,data,298,49.501661,758
153,based,296,49.169435,487
65,finance,284,47.176080,382
89,result,271,45.016611,321
161,study,248,41.196013,398
154,performance,247,41.029900,477


### 문헌 등장 비율 30% 이상 단어 확인

In [7]:
candidate_stopwords = freq_df[
    freq_df['doc_ratio'] >= 30
]

candidate_stopwords[
    ['word', 'doc_ratio', 'total_freq']
]

,word,doc_ratio,total_freq
8,model,81.395349,1296
7,language,69.269103,649
83,large,67.441860,515
24,llm,60.963455,1312
310,data,49.501661,758
153,based,49.169435,487
65,finance,47.176080,382
89,result,45.016611,321
161,study,41.196013,398
154,performance,41.029900,477


## 1-4. 2차 전처리

### 단어 빈도 및 문헌 등장 비율에 따른 custom stopwords 정의

In [8]:
# Custom Stopwords 정의
custom_stopwords = {

    # 검색 query 영향 단어
    'llm',
    'large',
    'language',

    # 일반 학술 용어
    'study',
    'research',
    'paper',
    'method',
    'result',
    'approach',

    # 일반 표현
    'using',
    'based',

    # 논문 표현 부산물
    'et',
    'al'
}

# 최종 stopwords
final_stopwords = base_stopwords.union(custom_stopwords) # 위에서 정의한 custom stopwords를 최종 불용어에 추가

print("Custom Stopwords:")
print(sorted(custom_stopwords))

Custom Stopwords:
['al', 'approach', 'based', 'et', 'language', 'large', 'llm', 'method', 'paper', 'research', 'result', 'study', 'using']


### 최종 전처리

In [9]:
def final_clean_text(text):

    # 결측치 처리
    if pd.isna(text):
        return ""

    # 소문자 변환
    text = text.lower()

    # 영문자 외 제거
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)

    # 공백 정리
    text = re.sub(r'\s+', ' ', text).strip()

    # 토큰화
    words = text.split()

    # 최종 불용어 제거
    words = [
        w for w in words
        if w not in final_stopwords and len(w) > 2
    ]

    # 표제어 추출
    words = [lemmatizer.lemmatize(w) for w in words]

    # 문자열 결합
    return " ".join(words)

# 최종 전처리 적용
article_df['Cleaned_Abstract'] = article_df['Abstract'].apply(final_clean_text)

print(f"최종 전처리 완료: {len(article_df)}건")

# 결과 확인
article_df[['Title', 'Cleaned_Abstract']].head(5)

최종 전처리 완료: 602건


,Title,Cleaned_Abstract
0,The Odyssey of robots.txt Governance: Measurin...,web content essential element model service su...
1,Evaluation of a Large Language Model on the Am...,backgroundlarge model llm including chatgpt ch...
2,HDLGen-ChatGPT Case Study: RISC-V Processor VH...,present open source hdlgen chatgpt application...
3,Experience with Large Language Model Applicati...,model llm offer promising capability informati...
4,Large Language Model Agents for Investment Man...,recent advance model llm triggered new wave in...


### BERTopic에 사용할 데이터프레임 저장

In [10]:
# BERTopic에 사용할 컬럼만 저장
final_df = article_df[['Title', 'Cleaned_Abstract']]

# 빈 데이터 제거
final_df = final_df[
    final_df['Cleaned_Abstract'].str.strip() != ''
]

# CSV 저장
final_df.to_csv(
    'cleaned_finance_ai_abstracts.csv',
    index=False,
    encoding='utf-8-sig'
)

print("최종 전처리 데이터 저장 완료: cleaned_finance_ai_abstracts.csv")
print(final_df.head())

최종 전처리 데이터 저장 완료: cleaned_finance_ai_abstracts.csv
                                               Title  \
0  The Odyssey of robots.txt Governance: Measurin...   
1  Evaluation of a Large Language Model on the Am...   
2  HDLGen-ChatGPT Case Study: RISC-V Processor VH...   
3  Experience with Large Language Model Applicati...   
4  Large Language Model Agents for Investment Man...   

                                    Cleaned_Abstract  
0  web content essential element model service su...  
1  backgroundlarge model llm including chatgpt ch...  
2  present open source hdlgen chatgpt application...  
3  model llm offer promising capability informati...  
4  recent advance model llm triggered new wave in...  


# [2] Topic Modeling
# 여기부터는 Colab T4 GPU 환경에서 작업하였음

### BERTopic 및 라이브러리 설정

In [26]:
!pip install bertopic
!pip install sentence-transformers
!pip install umap-learn
!pip install hdbscan

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 5.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.7/588.7 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 661.5/661.5 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 4.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 4.9 MB/s eta 0:00:00a 0:00:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 4.8 MB/s eta 0:00:00a 0:00:01m
  Attempting uninstall: regex
    Found existing installation: regex 2024.9.11
    Uninstalling regex-2024.9.11:
      Successfully uninstalled regex-2024.9.11


In [ ]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from bertopic.vectorizers import ClassTfidfTransformer
from umap import UMAP
from hdbscan import HDBSCAN

## 2-1. BERTopic 모델링

In [ ]:
# BERTopic 모델링을 위한 문서 리스트 생성
docs = article_df['Cleaned_Abstract'].tolist()

# SentenceTransformer 모델
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

# UMAP 모델: 차원 축소
umap_model = UMAP(
    n_neighbors=15, 
    n_components=5,
    min_dist=0.0, 
    metric='cosine',     # 문장 임베딩에 적합
    random_state=42
)

# HDBSCAN 모델: 클러스터링
hdbscan_model = HDBSCAN(
    min_cluster_size=15,      # 602개 문헌: 5 → 너무 잘게 쪼개지고, 30 → 너무 큰 토픽만 남아서 최소 클러스터를 15로 설정
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True
)

# CountVectorizer 모델: BoW 벡터화
vectorizer_model = CountVectorizer(
    stop_words='english',
    ngram_range=(1, 2),      # unigram과 bigram 모두 고려 -> "risk management"과 같은 단어를 topic keyword 포착 가능
    min_df=5     # 최소 5개 문헌에 등장하는 단어만 고려 -> 너무 희귀한 단어는 노이즈로 작용할 수 있기 때문에 제거
)

# c-tf-idf 모델: 토픽별 중요 단어 계산
ctfidf_model = ClassTfidfTransformer()

### BERTopic 모델 생성

In [ ]:
topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    nr_topics=8,
    calculate_probabilities=True,
    verbose=True
)

## 2-2. 모델 학습

In [ ]:
topics, probs = topic_model.fit_transform(docs)

## 2-3. 결과 확인

In [ ]:
# 토픽 확인
print(f"토픽 정보: {topic_model.get_topic_info()}")

# 토픽 키워드 확인
print(f"토픽 0 키워드: {topic_model.get_topic(0)}")

In [ ]:
topic_model.visualize_topics()

In [ ]:
topic_model.visualize_barchart()

In [ ]:
topic_model.visualize_heatmap()